In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
candidate_paths = [
    "student_data_cleaned.csv",
    "StudentPerformanceFactors.csv",
    "student_data.csv",
    os.path.join("..", "data", "student_data_cleaned.csv"),
    os.path.join("data", "student_data_cleaned.csv")
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Clean dataset not found! Make sure 'student_data_cleaned.csv' is uploaded or in data/.")

df = pd.read_csv(data_path)

In [3]:
if df['Teacher_Quality'].isnull().sum() > 0:
    df['Teacher_Quality'] = df['Teacher_Quality'].fillna(df['Teacher_Quality'].mode()[0])
if df['Parental_Education_Level'].isnull().sum() > 0:
    df['Parental_Education_Level'] = df['Parental_Education_Level'].fillna(df['Parental_Education_Level'].mode()[0])
if df['Distance_from_Home'].isnull().sum() > 0:
    df['Distance_from_Home'] = df['Distance_from_Home'].fillna(df['Distance_from_Home'].mode()[0])
df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")

Loaded dataset: 6607 rows, 20 columns.


In [4]:
X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

In [5]:
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Total Predictors: {X.shape[1]}")
print(f"  - Numerical features ({len(num_features)}): {num_features}")
print(f"  - Categorical features ({len(cat_features)}): {cat_features}")
print(f"\nTarget Variable: Exam_Score (Shape: {y.shape})")

Total Predictors: 19
  - Numerical features (6): ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']
  - Categorical features (13): ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']

Target Variable: Exam_Score (Shape: (6607,))


In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f"Training set:   {X_train.shape[0]} rows (80%)")
print(f"Testing set:    {X_test.shape[0]} rows (20%)")

Training set:   5285 rows (80%)
Testing set:    1322 rows (20%)


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ],
    remainder='drop'
)

In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [9]:
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_processed_features = num_features + encoded_cat_names

In [10]:
X_train_df = pd.DataFrame(X_train_processed, columns=all_processed_features, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=all_processed_features, index=X_test.index)

print(f"Processed Training Matrix shape: {X_train_df.shape}")
print(f"Processed Testing Matrix shape:  {X_test_df.shape}")
print(f"\nTotal encoded features: {len(all_processed_features)}")
display(X_train_df.head(3))

Processed Training Matrix shape: (5285, 27)
Processed Testing Matrix shape:  (1322, 27)

Total encoded features: 27


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Parental_Involvement_Low,Parental_Involvement_Medium,Access_to_Resources_Low,Access_to_Resources_Medium,...,Teacher_Quality_Medium,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Learning_Disabilities_Yes,Parental_Education_Level_High School,Parental_Education_Level_Postgraduate,Distance_from_Home_Moderate,Distance_from_Home_Near,Gender_Male
5810,1.173579,-0.075846,0.669094,-0.837277,0.405784,1.964909,1.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1268,-0.661983,0.531894,-0.016740,1.314728,0.405784,0.030381,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
414,0.339232,0.618714,0.669094,0.551113,-0.405017,-1.904146,1.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [11]:
output_dir = os.path.join("..", "data") if os.path.exists("..") and os.path.exists(os.path.join("..", "data")) else "data"
os.makedirs(output_dir, exist_ok=True)

X_train_df.to_csv(os.path.join(output_dir, "X_train.csv"), index=False)
X_test_df.to_csv(os.path.join(output_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(output_dir, "y_test.csv"), index=False)

print(f"✓ Successfully exported processed splits to '{output_dir}/':")
print(f"  - X_train.csv: {X_train_df.shape}")
print(f"  - X_test.csv:  {X_test_df.shape}")
print(f"  - y_train.csv: {y_train.shape}")
print(f"  - y_test.csv:  {y_test.shape}")

✓ Successfully exported processed splits to 'data/':
  - X_train.csv: (5285, 27)
  - X_test.csv:  (1322, 27)
  - y_train.csv: (5285,)
  - y_test.csv:  (1322,)


## Feature Selection vs. Feature Engineering

* **Feature Selection**: The deliberate process of evaluating and selecting a subset of the most relevant, non-redundant existing features to include in the model. The goal is to reduce dimensionality, prevent overfitting, eliminate arbitrary noise (e.g., student IDs), and ensure zero target leakage.
* **Feature Engineering**: The process of creating new features from existing raw data (e.g., interaction terms, polynomial terms, binning continuous variables) or transforming representations (e.g., one-hot encoding categorical variables, standard scaling numeric variables).

### Selection Criteria for this System:
1. **Domain Relevance**: Does the feature directly reflect student academic engagement or learning environment?
2. **Predictive Utility**: Did correlation ($r$) and ANOVA tests establish a statistically significant relationship with `Exam_Score`?
3. **Information Leakage**: Does the feature reveal post-exam outcomes? (None detected).
4. **Redundancy**: Does the feature cause multicollinearity? (Confirmed all pairwise $\vert{}r\vert{} < 0.05$).

### Feature Selection Justification Matrix

| Feature Name | Data Type | Selected? | Selection Rationale / Domain Justification |
| :--- | :--- | :---: | :--- |
| **Attendance** | Numerical | **Yes** | Primary academic driver; exhibits strongest linear correlation ($r = 0.58$) with final exam marks. |
| **Hours_Studied** | Numerical | **Yes** | Second strongest driver ($r = 0.45$); directly quantifies independent study effort. |
| **Previous_Scores** | Numerical | **Yes** | Academic baseline indicator ($r = 0.18$); controls for prior subject aptitude. |
| **Tutoring_Sessions** | Numerical | **Yes** | Supplemental instructional support ($r = 0.16$); represents guided intervention. |
| **Parental_Involvement** | Categorical | **Yes** | Statistically significant score variation via ANOVA ($p < 0.001$); measures home learning support. |
| **Access_to_Resources** | Categorical | **Yes** | Statistically significant ($p < 0.001$); separates students by study material and device availability. |
| **Motivation_Level** | Categorical | **Yes** | Intrinsic psychological factor; demonstrates distinct score variance across levels. |
| **Teacher_Quality** | Categorical | **Yes** | Institutional quality indicator; captures classroom instruction efficacy. |
| **School_Type** | Categorical | **Yes** | Structural attribute (Public vs. Private); distinguishes institutional resource disparities. |
| **Peer_Influence** | Categorical | **Yes** | Social environment factor; accounts for positive or negative study group habits. |
| **Learning_Disabilities**| Categorical | **Yes** | Essential indicator for individual learning accommodations and needs. |
| **Parental_Education_Level** | Categorical | **Yes** | Established socioeconomic background proxy linked to academic attainment. |
| **Distance_from_Home** | Categorical | **Yes** | Logistical factor; long travel times impact student fatigue and available study hours. |
| **Internet_Access** | Categorical | **Yes** | Core digital access metric required for modern online learning and assignments. |
| **Extracurricular_Activities** | Categorical | **Yes** | Behavioral balance indicator; balances extracurricular engagement against academic time. |
| **Family_Income** | Categorical | **Yes** | Socioeconomic proxy that influences tutoring access, books, and study environments. |
| **Gender** | Categorical | **Yes** | Demographic feature retained for subgroup fairness evaluation and bias auditing. |
| **Physical_Activity** | Numerical | **Yes** | Retained for holistic lifestyle profiling despite weak linear correlation ($r \approx 0.03$). |
| **Sleep_Hours** | Numerical | **Yes** | Retained as a baseline health metric despite near-zero linear correlation ($r \approx -0.02$). |
| **Student_ID** *(if present)* | Identifier | **No (Dropped)** | Arbitrary database index with zero predictive value; dropped to prevent memorization and leakage. |